In [4]:
import os
os.environ["PDF_FOLDER_HINT"] = "/content/drive/MyDrive/lawing"

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# %% [markdown]
# Kaggle All-in-One Legal Assistant
#
# What this file does:
# 1. Auto-installs required packages if they are missing
# 2. Finds the PDF dataset you added to Kaggle
# 3. Builds the vector database if it does not exist yet
# 4. Saves artifacts under /kaggle/working so Kaggle keeps them as notebook output
# 5. Launches a Gradio UI where the user must enter their own API key
#
# Typical Kaggle usage:
# - Add your PDF dataset
# - Copy or run this file
# - Run it top to bottom
# - Use the Gradio UI and paste your API key there

# %%
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from pathlib import Path
from threading import Lock
import argparse
import glob
import importlib.util
import inspect
import json
import os
import pickle
import re
import shutil
import subprocess
import sys
import time
from typing import Dict, Iterable, List, Tuple


# %% [markdown]
# Cell 1: Configuration

# %%

# ── Environment detection ─────────────────────────────────────────────────────
# Runs before anything else so every path and flag below can branch correctly.
def _detect_env() -> str:
    """Return 'colab', 'kaggle', or 'local'."""
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.path.isdir("/kaggle/working"):
        return "kaggle"
    return "local"

ENV = _detect_env()
print(f"Environment detected: {ENV}")

# ── GPU detection (needed by package selection and device assignment below) ───
def _detect_gpu() -> bool:
    """Return True if nvidia-smi exits cleanly — that's sufficient proof of a GPU."""
    try:
        subprocess.check_output(["nvidia-smi"], stderr=subprocess.STDOUT)
        return True
    except Exception:
        return False

HAS_GPU = _detect_gpu()
print(f"GPU available: {HAS_GPU}")

AUTO_INSTALL_PACKAGES = True
DEFAULT_PROVIDER = "openai"
DEFAULT_MODEL_NAME = ""
ALLOWED_PROVIDERS = ("openai", "groq", "gemini")
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"
CHUNK_SIZE = 300
OVERLAP = 50
BATCH_SIZE = 64 if HAS_GPU else 32   # larger batches are free on GPU
TOP_K = 3
MIN_RELEVANCE_SCORE = 0.30
MAX_QUERY_CHARS = 1800
MAX_HISTORY_TURNS = 4
MAX_CONTEXT_WORDS = 500    # FIX (answer cutoff): was 250 — too little context starved the model
MAX_OUTPUT_TOKENS = 1200   # FIX (answer cutoff): was 350 — sentences were cut mid-way
REQUESTS_PER_MINUTE = 8
MIN_REQUEST_INTERVAL_SECONDS = 2.5
QUEUE_MAX_SIZE = 16
DEFAULT_CONCURRENCY_LIMIT = 2
LAUNCH_SHARE_LINK = True

# ── Platform-specific paths ───────────────────────────────────────────────────
if ENV == "kaggle":
    KAGGLE_INPUT_ROOT  = Path("/kaggle/input")
    _WORKING_ROOT      = Path("/kaggle/working")
    PDF_SEARCH_ROOTS   = [KAGGLE_INPUT_ROOT]
elif ENV == "colab":
    KAGGLE_INPUT_ROOT  = Path("/content")          # Colab mount / upload target
    _WORKING_ROOT      = Path("/content/legal_rag")
    _WORKING_ROOT.mkdir(parents=True, exist_ok=True)
    PDF_SEARCH_ROOTS   = [Path("/content")]
else:                                               # local dev / CI
    KAGGLE_INPUT_ROOT  = Path(".")
    _WORKING_ROOT      = Path("./legal_rag_working")
    _WORKING_ROOT.mkdir(parents=True, exist_ok=True)
    PDF_SEARCH_ROOTS   = [Path(".")]

PDF_FOLDER_HINT = os.getenv("PDF_FOLDER_HINT", "").strip()

# Colab: fix the artifact path so the download cell always works with the
# same path regardless of how _WORKING_ROOT resolved.
if ENV == "colab":
    ARTIFACT_DIR = Path("/content/legal_rag/legal_rag_artifacts")
else:
    ARTIFACT_DIR = Path(os.getenv("LOCAL_VECTOR_DIR", str(_WORKING_ROOT / "legal_rag_artifacts")))

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_OUTPUT_PATH = _WORKING_ROOT / "legal_rag_artifacts.zip"

PROVIDER_SPECS = {
    "openai": {
        "label": "OpenAI",
        "base_url": "https://api.openai.com/v1",
    },
    "groq": {
        "label": "Groq",
        "base_url": "https://api.groq.com/openai/v1",
    },
    "gemini": {
        "label": "Gemini (Google)",
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai",
    },
}


# %% [markdown]
# Cell 2: Package bootstrap

# %%
class UserFacingError(RuntimeError):
    """Safe exception whose message can be shown in the UI."""

# Packages that MUST be present for the app to function at all.
# NOTE: torch is listed FIRST so pip installs the right wheels (CPU or CUDA)
# before sentence-transformers resolves torch as a dependency.  If torch were
# listed after sentence-transformers, pip would install CPU torch as a
# transitive dep and then immediately try to clobber it with the CUDA build.
_REQUIRED_PACKAGES: dict[str, str] = {
    "torch": (
        "torch --index-url https://download.pytorch.org/whl/cu121"
        if HAS_GPU else "torch"
    ),
    "gradio": "gradio>=4.0.0",
    "numpy": "numpy>=1.24.0",
    "requests": "requests>=2.28.0",
    "sentence_transformers": "sentence-transformers>=2.2.0",
}

# Optional PDF extractors — the app degrades gracefully when some are absent.
_OPTIONAL_PDF_PACKAGES: dict[str, str] = {
    "fitz": "pymupdf>=1.23.0",
    "pdfplumber": "pdfplumber>=0.10.0",
    "pypdf": "pypdf>=3.0.0",
}

_INTERNET_TEST_HOST    = "pypi.org"
_INTERNET_TEST_PORT    = 443
_INTERNET_TEST_TIMEOUT = 5    # seconds — fail fast instead of pip's 4-retry hang
_PIP_TIMEOUT           = 60   # seconds per package
_PIP_RETRIES           = 1    # one retry max; surface errors quickly


def _internet_is_available() -> bool:
    """Return True if a TCP connection to PyPI can be opened within the timeout."""
    import socket
    try:
        socket.setdefaulttimeout(_INTERNET_TEST_TIMEOUT)
        with socket.create_connection((_INTERNET_TEST_HOST, _INTERNET_TEST_PORT)):
            return True
    except OSError:
        return False


def _pip_install(package: str, *, quiet: bool = True) -> bool:
    """
    Install a single pip package. Returns True on success, False on failure.
    --retries and --timeout prevent the multi-minute silent hang that the
    original code produced when internet was off.
    """
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--retries", str(_PIP_RETRIES),
        "--timeout", str(_PIP_TIMEOUT),
        *(("-q",) if quiet else ()),
        *package.split(),   # allows extra flags like --index-url in the string
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0


def _abort_no_internet(missing_required: list[str]) -> None:
    banner = "=" * 64
    print(f"\n{banner}")
    print("  INSTALL FAILED — No internet access detected.")
    print()
    print("  Packages still needed:")
    for pkg in missing_required:
        print(f"    • {pkg.split()[0]}")   # strip extra pip flags for readability
    print()
    if ENV == "kaggle":
        print("  How to fix in Kaggle:")
        print("    1. Open the right-hand Settings panel.")
        print("    2. Under 'Internet', switch it ON.")
        print("    3. Re-run this cell.")
    elif ENV == "colab":
        print("  How to fix in Colab:")
        print("    Colab should always have internet. If this fires, the PyPI")
        print("    connectivity test timed out — try re-running the cell.")
    else:
        print("  Check your network connection and re-run.")
    print(f"{banner}\n")
    raise UserFacingError(
        "Internet access is off. Enable it in Notebook Settings → Internet and re-run."
    )


def ensure_packages() -> None:
    if not AUTO_INSTALL_PACKAGES:
        return

    # Always use faiss-cpu.
    # faiss-gpu on PyPI is compiled for CUDA 11.x; Colab runs CUDA 12.x so it
    # installs silently then crashes on `import faiss`.  For RAG corpus sizes
    # (tens of thousands of chunks) a CPU flat index completes searches in <1ms
    # — not the bottleneck.  GPU acceleration is used where it matters: encoding.
    required = {**_REQUIRED_PACKAGES, "faiss": "faiss-cpu"}

    # ── Which packages are actually missing? ─────────────────────────────────
    missing_required = [
        pkg for mod, pkg in required.items()
        if importlib.util.find_spec(mod) is None
    ]
    missing_optional = [
        pkg for mod, pkg in _OPTIONAL_PDF_PACKAGES.items()
        if importlib.util.find_spec(mod) is None
    ]

    if not missing_required and not missing_optional:
        print("All packages already installed.")
        return

    print(f"Packages to install: {len(missing_required)} required, {len(missing_optional)} optional")

    # ── Pre-flight connectivity check — fail in <5 s instead of ~2 min ───────
    if not _internet_is_available():
        if missing_required:
            _abort_no_internet(missing_required)
        else:
            print(
                "WARNING: Internet appears unavailable. "
                "Optional PDF extractors could not be installed — "
                "the app will use whichever extractors are already present."
            )
            return

    # ── Install required packages (hard fail if any cannot be installed) ──────
    failed_required: list[str] = []
    for pkg in missing_required:
        label = pkg.split()[0]   # just package name for display
        print(f"  Installing (required): {label} ...", end=" ", flush=True)
        if _pip_install(pkg):
            print("OK")
        else:
            print("FAILED")
            failed_required.append(pkg)

    if failed_required:
        print("\n" + "=" * 64)
        print("  ERROR: Could not install required packages:")
        for pkg in failed_required:
            print(f"    • {pkg.split()[0]}")
        print("  Check your internet connection and try again.")
        print("=" * 64 + "\n")
        raise UserFacingError(
            f"Failed to install: {', '.join(p.split()[0] for p in failed_required)}. "
            "Check internet access and re-run."
        )

    # ── Install optional PDF extractors (soft fail — log and continue) ────────
    for pkg in missing_optional:
        label = pkg.split()[0]
        print(f"  Installing (optional): {label} ...", end=" ", flush=True)
        if _pip_install(pkg):
            print("OK")
        else:
            print("skipped (will use other available extractors)")


ensure_packages()

import gradio as gr
import numpy as np
import requests

try:
    import fitz  # type: ignore
except Exception:
    fitz = None

try:
    import pdfplumber  # type: ignore
except Exception:
    pdfplumber = None

try:
    from pypdf import PdfReader  # type: ignore
except Exception:
    PdfReader = None


# %% [markdown]
# Cell 3: Helpers and security
#
# NOTE: `UserFacingError` was moved to Cell 2 so `ensure_packages` can raise it.

# %%
def sanitize_text(value: str | None, max_chars: int) -> str:
    text = (value or "").replace("\x00", " ").strip()
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    if len(text) > max_chars:
        text = text[:max_chars].rstrip()
    return text


def redact_sensitive(text: str) -> str:
    redacted = text
    patterns = [
        r"sk-[A-Za-z0-9_-]{12,}",
        r"hf_[A-Za-z0-9]{12,}",
        r"gsk_[A-Za-z0-9_-]{12,}",
        r"AIza[A-Za-z0-9_-]{35}",
    ]
    for pattern in patterns:
        redacted = re.sub(pattern, "[REDACTED]", redacted)
    return redacted


def safe_error_message(exc: Exception) -> str:
    if isinstance(exc, UserFacingError):
        return str(exc)
    detail = redact_sensitive(str(exc)).strip()
    if detail:
        return f"Something went wrong. Internal detail: {detail[:180]}"
    return "Something went wrong."


def normalize_rows(values: np.ndarray) -> np.ndarray:
    array = np.asarray(values, dtype=np.float32)
    if array.ndim == 1:
        array = array.reshape(1, -1)
    norms = np.linalg.norm(array, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return array / norms


def detect_intent(query: str) -> str:
    query_lower = query.lower().strip()
    legal_indicators = [
        # Acts and codes
        "section", "act", "law", "ipc", "bns", "bnss", "bnns", "pocso", "ndps",
        "crpc", "it act", "constitution", "article", "schedule", "clause",
        # People and status
        "accused", "victim", "complainant", "witness", "juvenile", "minor",
        "child", "abscond", "detain", "detention", "custody", "prisoner",
        # Proceedings
        "court", "tribunal", "magistrate", "judge", "appeal", "petition",
        "bail", "arrest", "warrant", "summons", "charge", "trial", "acquit",
        # Liability and culpability
        "criminal", "civil", "liable", "liability", "culpable", "negligence",
        "intent", "mens rea", "actus reus", "abetment", "conspiracy",
        # Outcomes
        "penalty", "offense", "offence", "crime", "rights", "punishment",
        "sentence", "fine", "imprisonment", "provisions", "legal", "illegal",
        "exempt", "immunity", "pardon", "remission",
    ]
    if any(indicator in query_lower for indicator in legal_indicators):
        return "LEGAL"
    if re.search(r"\b(section|sec|article|art)\s*\d+", query_lower):
        return "LEGAL"
    if len(query_lower.split()) <= 3:
        return "CONVERSATIONAL"
    return "CONVERSATIONAL"


def history_to_messages(history: list | None, max_turns: int, max_chars: int) -> list[dict[str, str]]:
    if not history:
        return []

    normalized: list[dict[str, str]] = []
    for item in history:
        if isinstance(item, dict):
            role = str(item.get("role", "")).strip().lower()
            content = item.get("content")
            if role in {"user", "assistant"} and isinstance(content, str):
                clean_content = sanitize_text(content, max_chars)
                if clean_content:
                    normalized.append({"role": role, "content": clean_content})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_text = sanitize_text(str(item[0] or ""), max_chars)
            assistant_text = sanitize_text(str(item[1] or ""), max_chars)
            if user_text:
                normalized.append({"role": "user", "content": user_text})
            if assistant_text:
                normalized.append({"role": "assistant", "content": assistant_text})

    return normalized[-(max_turns * 2):]


@dataclass(frozen=True)
class RateLimitPolicy:
    requests_per_minute: int
    min_interval_seconds: float


class SessionRateLimiter:
    def __init__(self, policy: RateLimitPolicy) -> None:
        self.policy = policy
        self._events: dict[str, deque[float]] = {}
        self._lock = Lock()

    def check(self, session_id: str) -> None:
        if not session_id:
            return

        now = time.monotonic()
        with self._lock:
            bucket = self._events.setdefault(session_id, deque())
            while bucket and now - bucket[0] > 60:
                bucket.popleft()

            if bucket and now - bucket[-1] < self.policy.min_interval_seconds:
                wait_time = self.policy.min_interval_seconds - (now - bucket[-1])
                raise UserFacingError(f"Please wait {wait_time:.1f}s before sending another request.")

            if len(bucket) >= self.policy.requests_per_minute:
                raise UserFacingError("Too many requests from this session. Pause for a minute and try again.")

            bucket.append(now)


# %% [markdown]
# Cell 4: PDF extraction and chunking

# %%
def extract_text_pymupdf(pdf_path: str) -> Tuple[str, str]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")
    document = fitz.open(pdf_path)
    text = ""
    for page in document:
        text += page.get_text()
    document.close()
    return text, "pymupdf"


def extract_text_pdfplumber(pdf_path: str) -> Tuple[str, str]:
    if pdfplumber is None:
        raise RuntimeError("pdfplumber is not installed.")
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text, "pdfplumber"


def extract_text_pypdf(pdf_path: str) -> Tuple[str, str]:
    if PdfReader is None:
        raise RuntimeError("pypdf is not installed.")
    text = ""
    with open(pdf_path, "rb") as file_handle:
        reader = PdfReader(file_handle)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text, "pypdf"


def find_pdf_folder() -> Path:
    if PDF_FOLDER_HINT and Path(PDF_FOLDER_HINT).exists():
        return Path(PDF_FOLDER_HINT)

    candidates: list[tuple[int, int, Path]] = []
    for search_root in PDF_SEARCH_ROOTS:
        if not search_root.exists():
            continue
        for directory, _, _ in os.walk(search_root):
            folder = Path(directory)
            pdf_count = len(list(folder.glob("*.pdf")))
            if pdf_count > 0:
                score = pdf_count
                lower_path = str(folder).lower()
                if "lawing" in lower_path:
                    score += 100
                if "legal" in lower_path:
                    score += 50
                candidates.append((score, pdf_count, folder))

    if not candidates:
        # Show the upload tip here, only when PDFs are actually missing.
        if ENV == "colab":
            print(
                "\nTip (Colab): upload your PDFs with:\n"
                "  from google.colab import files; files.upload()\n"
                "or mount Drive and set PDF_FOLDER_HINT to the folder path.\n"
            )
        hint = (
            "/kaggle/input" if ENV == "kaggle"
            else "/content (upload via files.upload() or mount Drive)"
            if ENV == "colab"
            else "the current directory"
        )
        raise UserFacingError(
            f"No PDF files were found under {hint}. Add your PDF dataset first."
        )

    candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
    selected = candidates[0][2]
    print(f"Using PDF folder: {selected}")
    return selected


def extract_all_pdfs(pdf_folder: Path) -> Dict[str, str]:
    methods = [extract_text_pymupdf, extract_text_pdfplumber, extract_text_pypdf]
    pdf_texts: Dict[str, str] = {}
    pdf_files = sorted(glob.glob(str(pdf_folder / "*.pdf")))
    print(f"Found {len(pdf_files)} PDFs in {pdf_folder}")

    for pdf_path in pdf_files:
        filename = os.path.basename(pdf_path)
        print(f"Processing: {filename}")
        extracted = False
        for method in methods:
            try:
                text, method_name = method(pdf_path)
                if text and len(text.strip()) > 100:
                    pdf_texts[filename] = text
                    print(f"  Extracted using {method_name} ({len(text)} chars)")
                    extracted = True
                    break
            except Exception:
                continue
        if not extracted:
            print(f"  Failed to extract: {filename}")

    return pdf_texts


class LegalDocumentChunker:
    def __init__(self, chunk_size: int = 300, overlap: int = 50):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.section_pattern = re.compile(r"(?:Section|SECTION|Sec\.|SEC\.)\s*(\d+[A-Z]?)", re.IGNORECASE)
        self.article_pattern = re.compile(r"(?:Article|ARTICLE|Art\.|ART\.)\s*(\d+[A-Z]?)", re.IGNORECASE)

    def extract_act_name(self, text: str, filename: str) -> str:
        first_part = text[:500].upper()
        patterns = [r"(.*?ACT,?\s*\d{4})", r"THE\s+(.*?)\s+ACT", r"(.*?)\s+CODE"]
        for pattern in patterns:
            match = re.search(pattern, first_part)
            if match:
                return match.group(1).strip()
        return filename.replace(".pdf", "").replace("_", " ").title()

    def detect_section_boundaries(self, text: str) -> List[Tuple[int, str, str]]:
        boundaries = []
        for match in self.section_pattern.finditer(text):
            boundaries.append((match.start(), "Section", match.group(1)))
        for match in self.article_pattern.finditer(text):
            boundaries.append((match.start(), "Article", match.group(1)))
        boundaries.sort(key=lambda item: item[0])
        return boundaries

    def clean_text(self, text: str) -> str:
        cleaned = re.sub(r"[ \t]+", " ", text)
        cleaned = re.sub(r"\n\s*\n\s*\n+", "\n\n", cleaned)
        cleaned = re.sub(r"\n\s*\d+\s*\n", "\n", cleaned)
        cleaned = re.sub(r"https?://\S+", "", cleaned)
        return cleaned.strip()

    def create_semantic_chunks(self, text: str, act_name: str, filename: str) -> List[Dict]:
        cleaned_text = self.clean_text(text)
        boundaries = self.detect_section_boundaries(cleaned_text)
        if not boundaries:
            return self._fallback_chunking(cleaned_text, act_name, filename)

        chunks: List[Dict] = []
        for i, (pos, marker_type, marker_num) in enumerate(boundaries):
            end_pos = boundaries[i + 1][0] if i < len(boundaries) - 1 else len(cleaned_text)
            section_text = cleaned_text[pos:end_pos].strip()
            if len(section_text.split()) > self.chunk_size:
                sub_chunks = self._split_long_section(section_text)
                for j, sub_chunk in enumerate(sub_chunks):
                    chunks.append(
                        {
                            "text": sub_chunk,
                            "metadata": {
                                "source_file": filename,
                                "act_name": act_name,
                                "section_type": marker_type,
                                "section_number": marker_num,
                                "sub_chunk_id": j,
                                "chunk_type": "section_split",
                            },
                        }
                    )
            else:
                chunks.append(
                    {
                        "text": section_text,
                        "metadata": {
                            "source_file": filename,
                            "act_name": act_name,
                            "section_type": marker_type,
                            "section_number": marker_num,
                            "chunk_type": "full_section",
                        },
                    }
                )
        return chunks

    def _split_long_section(self, text: str) -> List[str]:
        words = text.split()
        chunks = []
        step = max(1, self.chunk_size - self.overlap)
        for i in range(0, len(words), step):
            chunk = " ".join(words[i: i + self.chunk_size]).strip()
            if chunk:
                chunks.append(chunk)
            if i + self.chunk_size >= len(words):
                break
        return chunks

    def _fallback_chunking(self, text: str, act_name: str, filename: str) -> List[Dict]:
        words = text.split()
        chunks = []
        step = max(1, self.chunk_size - self.overlap)
        for i in range(0, len(words), step):
            chunk_text = " ".join(words[i: i + self.chunk_size]).strip()
            if chunk_text:
                chunks.append(
                    {
                        "text": chunk_text,
                        "metadata": {
                            "source_file": filename,
                            "act_name": act_name,
                            "chunk_id": i // step,
                            "chunk_type": "sliding_window",
                        },
                    }
                )
            if i + self.chunk_size >= len(words):
                break
        return chunks

    def add_context_to_chunks(self, chunks: List[Dict]) -> List[Dict]:
        for chunk in chunks:
            metadata = chunk["metadata"]
            context_prefix = f"[{metadata['act_name']}] "
            if "section_number" in metadata:
                context_prefix += f"{metadata['section_type']} {metadata['section_number']}: "
            chunk["enhanced_text"] = context_prefix + chunk["text"]
        return chunks


# %% [markdown]
# Cell 5: Embeddings, vector storage, and retrieval

# %%
class SentenceTransformerEmbedder:
    def __init__(self, model_name: str) -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore
        import torch

        # Auto-select the best available device.
        # Previously hard-coded to "cpu" which wasted GPU memory bandwidth.
        if HAS_GPU and torch.cuda.is_available():
            self._device = "cuda"
        else:
            self._device = "cpu"
        print(f"Embedding model device: {self._device}")
        self.model = SentenceTransformer(model_name, device=self._device)

    def encode(self, texts: list[str]) -> np.ndarray:
        return np.asarray(
            self.model.encode(
                texts,
                show_progress_bar=False,
                batch_size=BATCH_SIZE,
                convert_to_numpy=True,
            ),
            dtype=np.float32,
        )


class NumpyVectorIndex:
    def __init__(self, embeddings: np.ndarray) -> None:
        self.embeddings = normalize_rows(np.asarray(embeddings, dtype=np.float32))

    def search(self, query_embedding: np.ndarray, top_k: int) -> tuple[np.ndarray, np.ndarray]:
        query = normalize_rows(query_embedding)
        scores = np.matmul(query, self.embeddings.T)
        indices = np.argsort(-scores, axis=1)[:, :top_k]
        top_scores = np.take_along_axis(scores, indices, axis=1)
        return top_scores.astype(np.float32), indices.astype(np.int64)


class OptimizedLegalRetriever:
    def __init__(
        self,
        embedding_model: SentenceTransformerEmbedder,
        vector_index,
        all_chunks: list[str],
        chunk_metadata: list[dict],
    ) -> None:
        self.embedding_model = embedding_model
        self.index = vector_index
        self.all_chunks = all_chunks
        self.chunk_metadata = chunk_metadata
        self.query_prefix = "Represent this query for retrieving relevant legal passages: "
        # FIX 3: Cache whether the index is a native FAISS index (has an `ntotal`
        #         attribute) vs. our NumpyVectorIndex so we don't rely on isinstance
        #         with an optional import.
        self._is_faiss = hasattr(vector_index, "ntotal")

    def preprocess_query(self, query: str) -> str:
        expansions = {
            r"\bBNS\b": "Bharatiya Nyaya Sanhita",
            r"\bBNNS\b": "Bharatiya Nagarik Suraksha Sanhita",
            r"\bCrPC\b": "Criminal Procedure Code",
            r"\bPOCSO\b": "Protection of Children from Sexual Offences",
            r"\bIT Act\b": "Information Technology Act",
            r"\bSec\.?\b": "Section",
            r"\bArt\.?\b": "Article",
        }
        updated = query
        for abbreviation, full_text in expansions.items():
            updated = re.sub(abbreviation, full_text, updated, flags=re.IGNORECASE)
        return updated.strip()

    def retrieve(self, query: str, top_k: int = 3) -> list[dict]:
        processed_query = self.preprocess_query(query)
        query_embedding = self.embedding_model.encode([self.query_prefix + processed_query])
        query_embedding = normalize_rows(query_embedding)
        scores, indices = self.index.search(query_embedding, max(top_k * 2, top_k))

        candidates: list[dict] = []
        query_keywords = set(processed_query.lower().split())
        for score, index in zip(scores[0], indices[0]):
            idx = int(index)
            if 0 <= idx < len(self.all_chunks):
                chunk_text = self.all_chunks[idx]
                chunk_keywords = set(chunk_text.lower().split())
                overlap = len(query_keywords & chunk_keywords) / max(len(query_keywords), 1)
                candidates.append(
                    {
                        "chunk": chunk_text,
                        "metadata": self.chunk_metadata[idx],
                        "score": float(score) + (overlap * 0.15),
                        "index": idx,
                    }
                )

        candidates.sort(key=lambda item: item["score"], reverse=True)
        return candidates[:top_k]


def format_retrieved_context(chunks_data: list[dict], max_words_per_chunk: int) -> str:
    formatted_parts: list[str] = []
    for i, data in enumerate(chunks_data, 1):
        meta = data["metadata"]
        chunk = data["chunk"]
        words = chunk.split()
        if len(words) > max_words_per_chunk:
            chunk = " ".join(words[:max_words_per_chunk]) + " ..."

        source = meta.get("source_file", "Unknown").replace(".pdf", "")
        section_info = ""
        if "section_number" in meta:
            section_info = f" - {meta.get('section_type', 'Section')} {meta['section_number']}"

        formatted_parts.append(f"[Reference {i}: {source}{section_info}]\n{chunk}")
    return "\n\n---\n\n".join(formatted_parts)


def build_vector_bundle() -> dict:
    pdf_folder = find_pdf_folder()
    pdf_texts = extract_all_pdfs(pdf_folder)
    if not pdf_texts:
        raise UserFacingError("No text could be extracted from the PDFs. Check the dataset contents.")

    chunker = LegalDocumentChunker(chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    all_chunks: list[str] = []
    chunk_metadata: list[dict] = []

    for filename, text in pdf_texts.items():
        act_name = chunker.extract_act_name(text, filename)
        document_chunks = chunker.create_semantic_chunks(text, act_name, filename)
        document_chunks = chunker.add_context_to_chunks(document_chunks)
        for chunk in document_chunks:
            all_chunks.append(chunk["enhanced_text"])
            chunk_metadata.append(chunk["metadata"])
        print(f"Chunked {filename}: {len(document_chunks)} chunks")

    print(f"Total chunks: {len(all_chunks)}")
    embedder = SentenceTransformerEmbedder(EMBEDDING_MODEL_NAME)
    embeddings = []
    for start in range(0, len(all_chunks), BATCH_SIZE):
        batch = all_chunks[start: start + BATCH_SIZE]
        batch_embeddings = embedder.encode(batch)
        embeddings.extend(batch_embeddings)
        print(f"Embedded {start + len(batch)}/{len(all_chunks)} chunks")

    embeddings_array = normalize_rows(np.asarray(embeddings, dtype=np.float32))
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

    with (ARTIFACT_DIR / "chunks_data.pkl").open("wb") as file_handle:
        pickle.dump({"all_chunks": all_chunks, "chunk_metadata": chunk_metadata}, file_handle)
    np.save(ARTIFACT_DIR / "embeddings.npy", embeddings_array)

    faiss_created = False
    try:
        import faiss  # type: ignore

        # FIX 4: The original code called `faiss.index_gpu_to_cpu(index)` guarded
        #         only by `hasattr(faiss, "index_gpu_to_cpu")`.  That attribute
        #         exists in faiss-cpu too (it's a no-op there), so calling it on a
        #         CPU index would crash.  We now track whether the GPU transfer
        #         actually succeeded and only call the GPU→CPU conversion when
        #         `on_gpu` is True.
        cpu_index = faiss.IndexFlatIP(embeddings_array.shape[1])
        cpu_index.add(embeddings_array)

        on_gpu = False
        active_index = cpu_index
        try:
            res = faiss.StandardGpuResources()
            active_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
            on_gpu = True
            print("Hardware Acceleration: FAISS index moved to GPU for building")
        except Exception:
            print("Hardware Acceleration: Using CPU for FAISS indexing")

        # Always save the CPU version — GPU indexes cannot be serialised.
        saveable_index = faiss.index_gpu_to_cpu(active_index) if on_gpu else cpu_index
        faiss.write_index(saveable_index, str(ARTIFACT_DIR / "legal_faiss_index.index"))
        faiss_created = True
    except Exception as e:
        print(f"Note: FAISS index creation skipped ({e}). Falling back to Numpy search.")
        faiss_created = False

    manifest = {
        "embedding_model_name": EMBEDDING_MODEL_NAME,
        "vector_count": len(all_chunks),
        "source_pdf_count": len(pdf_texts),
        "chunk_size": CHUNK_SIZE,
        "overlap": OVERLAP,
        "pdf_folder": str(pdf_folder),
        "files": {
            "chunks": "chunks_data.pkl",
            "embeddings": "embeddings.npy",
            "faiss": "legal_faiss_index.index" if faiss_created else None,
        },
    }
    with (ARTIFACT_DIR / "manifest.json").open("w", encoding="utf-8") as file_handle:
        json.dump(manifest, file_handle, indent=2)

    if ZIP_OUTPUT_PATH.exists():
        ZIP_OUTPUT_PATH.unlink()
    archive_root = str(ZIP_OUTPUT_PATH).removesuffix(".zip")
    shutil.make_archive(archive_root, "zip", root_dir=ARTIFACT_DIR)
    print(f"Saved artifact folder: {ARTIFACT_DIR}")
    if ENV == "kaggle":
        print(f"Saved Kaggle output zip: {ZIP_OUTPUT_PATH}")
    elif ENV == "colab":
        print(f"Saved zip: {ZIP_OUTPUT_PATH}")
        print("  To download in Colab run: from google.colab import files; files.download(str(ZIP_OUTPUT_PATH))")

    return {
        "manifest": manifest,
        "all_chunks": all_chunks,
        "chunk_metadata": chunk_metadata,
        "embeddings": embeddings_array,
        "embedder": embedder,
    }


def load_vector_bundle() -> dict:
    manifest_path = ARTIFACT_DIR / "manifest.json"
    chunks_path = ARTIFACT_DIR / "chunks_data.pkl"
    embeddings_path = ARTIFACT_DIR / "embeddings.npy"

    if not manifest_path.exists() or not chunks_path.exists() or not embeddings_path.exists():
        return build_vector_bundle()

    with manifest_path.open("r", encoding="utf-8") as file_handle:
        manifest = json.load(file_handle)
    with chunks_path.open("rb") as file_handle:
        data = pickle.load(file_handle)
    embeddings = np.load(embeddings_path)
    embedder = SentenceTransformerEmbedder(EMBEDDING_MODEL_NAME)

    return {
        "manifest": manifest,
        "all_chunks": list(data["all_chunks"]),
        "chunk_metadata": list(data["chunk_metadata"]),
        "embeddings": np.asarray(embeddings, dtype=np.float32),
        "embedder": embedder,
    }


def load_best_index(embeddings: np.ndarray):
    faiss_path = ARTIFACT_DIR / "legal_faiss_index.index"
    if faiss_path.exists():
        try:
            import faiss  # type: ignore

            index = faiss.read_index(str(faiss_path))
            try:
                res = faiss.StandardGpuResources()
                index = faiss.index_cpu_to_gpu(res, 0, index)
                print("Hardware Acceleration: FAISS index moved to GPU")
            except Exception:
                pass
            return index
        except Exception:
            pass
    return NumpyVectorIndex(embeddings)


# %% [markdown]
# Cell 6: Provider client and runtime

# %%
class MandatoryUserKeyProvider:
    def __init__(self) -> None:
        self.session = requests.Session()

    def stream_chat(
        self,
        provider_name: str,
        model_name: str,
        user_api_key: str,
        messages: list[dict[str, str]],
        *,
        temperature: float,
        max_tokens: int,
    ) -> Iterable[str]:
        provider_key = (provider_name or DEFAULT_PROVIDER).strip().lower()
        if provider_key not in ALLOWED_PROVIDERS:
            raise UserFacingError(
                f"Unsupported provider '{provider_name}'. Allowed providers: {', '.join(ALLOWED_PROVIDERS)}."
            )
        if provider_key not in PROVIDER_SPECS:
            raise UserFacingError(f"Provider '{provider_name}' is not configured.")

        api_key = sanitize_text(user_api_key, 500)
        if not api_key:
            raise UserFacingError("API key is required in the UI to use this app.")

        model = sanitize_text(model_name, 200)
        if not model:
            raise UserFacingError("Model name is required in the UI.")

        spec = PROVIDER_SPECS[provider_key]
        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "stream": True,
        }

        try:
            with self.session.post(
                f"{spec['base_url']}/chat/completions",
                headers=headers,
                json=payload,
                timeout=(10, 180),
                stream=True,
            ) as response:
                if response.status_code >= 400:
                    # Extract the raw message from the response body first.
                    raw_detail = ""
                    try:
                        error_payload = response.json()
                        # Some APIs return a JSON list instead of a dict — unwrap it.
                        if isinstance(error_payload, list):
                            error_payload = error_payload[0] if error_payload else {}
                        if isinstance(error_payload, dict):
                            err_obj = error_payload.get("error") or {}
                            if isinstance(err_obj, dict):
                                raw_detail = err_obj.get("message") or ""
                            else:
                                raw_detail = str(err_obj)
                            raw_detail = raw_detail or error_payload.get("message") or ""
                        else:
                            raw_detail = str(error_payload)
                    except ValueError:
                        raw_detail = response.text[:300]
                    raw_detail = redact_sensitive(str(raw_detail)).strip()

                    # Map status codes to clear, actionable messages.
                    status = response.status_code
                    if status == 401:
                        user_msg = (
                            f"❌ Invalid API key for {spec['label']}. "
                            "Check that you copied the full key with no extra spaces."
                        )
                    elif status == 403:
                        user_msg = (
                            f"❌ Access denied by {spec['label']} (403). "
                            "Your key does not have permission to use this model. "
                            "Check your plan or enable the model in your provider dashboard."
                        )
                    elif status == 404:
                        user_msg = (
                            f"❌ Model not found on {spec['label']} (404). "
                            f"The model name '{model}' is wrong or not available on your plan. "
                            "Check the exact model string in the provider docs."
                        )
                    elif status == 429:
                        user_msg = (
                            f"⏳ Rate limit / quota exceeded on {spec['label']} (429). "
                            "Wait a moment, check your usage limits, or upgrade your plan."
                        )
                    elif status == 500:
                        user_msg = (
                            f"🔥 {spec['label']} returned a server error (500). "
                            "This is on their side — wait a minute and retry."
                        )
                    elif status in (502, 503, 504):
                        user_msg = (
                            f"🔌 {spec['label']} is temporarily unavailable ({status}). "
                            "Try again in a few seconds."
                        )
                    else:
                        user_msg = (
                            f"❌ {spec['label']} rejected the request (HTTP {status})."
                        )

                    # Always append the raw provider message if it adds info.
                    if raw_detail and raw_detail.lower() not in user_msg.lower():
                        user_msg += f"\n\nProvider said: {raw_detail[:300]}"

                    raise UserFacingError(user_msg)

                for raw_line in response.iter_lines(decode_unicode=True):
                    if not raw_line:
                        continue
                    line = raw_line.strip()
                    if not line.startswith("data:"):
                        continue
                    data = line[5:].strip()
                    if data == "[DONE]":
                        break
                    try:
                        payload_chunk = json.loads(data)
                    except json.JSONDecodeError:
                        continue

                    # Some providers wrap the chunk in a list — unwrap it.
                    if isinstance(payload_chunk, list):
                        if not payload_chunk:
                            continue
                        payload_chunk = payload_chunk[0]
                    if not isinstance(payload_chunk, dict):
                        continue

                    choices = payload_chunk.get("choices")
                    if not choices or not isinstance(choices, list):
                        continue
                    first_choice = choices[0]
                    if not isinstance(first_choice, dict):
                        continue
                    delta = first_choice.get("delta", {})
                    if not isinstance(delta, dict):
                        continue
                    token = delta.get("content")
                    if token:
                        yield token
        except requests.RequestException as exc:
            raise UserFacingError(
                f"{spec['label']} request failed. Check the key, model, and internet setting."
            ) from exc


@dataclass
class AppRuntime:
    retriever: OptimizedLegalRetriever
    provider: MandatoryUserKeyProvider
    manifest: dict


def bootstrap_runtime() -> AppRuntime:
    bundle = load_vector_bundle()
    index = load_best_index(bundle["embeddings"])
    retriever = OptimizedLegalRetriever(
        embedding_model=bundle["embedder"],
        vector_index=index,
        all_chunks=bundle["all_chunks"],
        chunk_metadata=bundle["chunk_metadata"],
    )
    return AppRuntime(retriever=retriever, provider=MandatoryUserKeyProvider(), manifest=bundle["manifest"])


RUNTIME: AppRuntime | None = None
RUNTIME_ERROR: Exception | None = None
RATE_LIMITER = SessionRateLimiter(
    RateLimitPolicy(
        requests_per_minute=REQUESTS_PER_MINUTE,
        min_interval_seconds=MIN_REQUEST_INTERVAL_SECONDS,
    )
)

try:
    RUNTIME = bootstrap_runtime()
except Exception as exc:
    RUNTIME_ERROR = exc


# %% [markdown]
# Cell 7: Gradio app

# %%
def runtime_status_markdown() -> str:
    hw = "GPU ✓" if HAS_GPU else "CPU only"
    env_label = {"kaggle": "Kaggle", "colab": "Google Colab", "local": "Local"}.get(ENV, ENV)

    if RUNTIME is None:
        detail = safe_error_message(RUNTIME_ERROR) if RUNTIME_ERROR else "Runtime not initialized."
        return (
            "### Runtime Status\n"
            f"- Environment: `{env_label}` | Hardware: `{hw}`\n"
            "- Vector database: not ready\n"
            f"- Detail: `{detail}`"
        )

    source_count = RUNTIME.manifest.get("source_pdf_count", "unknown")
    vector_count = RUNTIME.manifest.get("vector_count", "unknown")
    return (
        "### Runtime Status\n"
        f"- Environment: `{env_label}` | Hardware: `{hw}`\n"
        "- Vector database: **ready** ✓\n"
        f"- Sources indexed: `{source_count}` PDFs\n"
        f"- Chunks loaded: `{vector_count}`\n"
        f"- Artifact folder: `{ARTIFACT_DIR}`\n"
        f"- Zip output: `{ZIP_OUTPUT_PATH}`"
    )


def build_settings_help() -> str:
    provider_lines = [f"- {PROVIDER_SPECS[name]['label']}" for name in ALLOWED_PROVIDERS]
    return (
        "### UI Rules\n"
        "- Enter your own API key in the UI. It is required.\n"
        "- The app does not use any owner or server-side fallback key.\n"
        "- Your key is only used for the current session path and is not saved by this code.\n"
        "- If the vector database exists, legal queries use grounded retrieval.\n"
        "- If the vector database build failed, you can still test the API key and model from the UI.\n\n"
        "### Allowed Providers\n" + "\n".join(provider_lines)
    )


def build_conversational_messages(message: str, history: list) -> list[dict[str, str]]:
    trimmed_history = history_to_messages(history, MAX_HISTORY_TURNS, MAX_QUERY_CHARS)
    return [
        {
            "role": "system",
            "content": "You are a helpful Indian legal AI assistant. For casual conversation, reply briefly and clearly in 2-3 sentences.",
        },
        *trimmed_history,
        {"role": "user", "content": message},
    ]


def build_legal_messages(query: str, chunks_data: list[dict]) -> list[dict[str, str]]:
    context = format_retrieved_context(chunks_data, MAX_CONTEXT_WORDS)
    return [
        {
            "role": "system",
            "content": "You are an expert on Indian law. Answer only from the provided legal references. Cite sections, acts, and provisions where possible. If the context is incomplete, say so plainly.",
        },
        {
            "role": "user",
            "content": (
                "Legal References:\n"
                f"{context}\n\n"
                f"Question: {query}\n\n"
                "Provide a grounded answer based only on the references above."
            ),
        },
    ]


def sources_status(chunks_data: list[dict], retrieval_time: float) -> str:
    lines = [f"Retrieved {len(chunks_data)} grounded references in {retrieval_time:.2f}s", "", "Sources:"]
    for chunk in chunks_data:
        source = chunk["metadata"].get("source_file", "Unknown").replace(".pdf", "")
        lines.append(f"- {source} ({chunk['score']:.0%} relevance)")
    return "\n".join(lines)


def stream_provider_reply(
    provider_name: str,
    model_name: str,
    user_api_key: str,
    messages: list[dict[str, str]],
    *,
    temperature: float,
):
    provider = MandatoryUserKeyProvider()
    full_text = ""
    for token in provider.stream_chat(
        provider_name,
        model_name,
        user_api_key,
        messages,
        temperature=temperature,
        max_tokens=MAX_OUTPUT_TOKENS,
    ):
        full_text += token
        yield full_text.strip()


def test_api_connection(provider_name: str, model_name: str, user_api_key: str) -> str:
    try:
        messages = [
            {"role": "system", "content": "Reply in one short sentence confirming the API connection is working."},
            {"role": "user", "content": "Say: API connection successful."},
        ]
        sample = ""
        for partial in stream_provider_reply(provider_name, model_name, user_api_key, messages, temperature=0.1):
            sample = partial
            if len(sample) >= 120:
                break
        return (
            f"Connection OK. Sample reply: {sample[:160]}"
            if sample
            else "Connection reached the provider, but no text was returned."
        )
    except Exception as exc:
        return safe_error_message(exc)


def chat(
    message: str,
    history: list,
    provider_name: str,
    model_name: str,
    user_api_key: str,
    request: gr.Request,
):
    try:
        session_id = request.session_hash if request and request.session_hash else "anonymous"
        RATE_LIMITER.check(session_id)

        clean_message = sanitize_text(message, MAX_QUERY_CHARS)
        if not clean_message:
            raise UserFacingError("Please enter a legal question or message.")

        intent = detect_intent(clean_message)
        if intent == "CONVERSATIONAL":
            yield "Thinking ..."
            messages = build_conversational_messages(clean_message, history)
            for partial in stream_provider_reply(
                provider_name, model_name, user_api_key, messages, temperature=0.6
            ):
                yield partial
            return

        if RUNTIME is None:
            detail = safe_error_message(RUNTIME_ERROR) if RUNTIME_ERROR else "Vector database not ready."
            yield (
                "The vector database is not ready yet, so grounded legal answers cannot run.\n\n"
                "You can still use the `Test API Key / Model` button to validate your provider setup.\n\n"
                f"Detail: {detail}"
            )
            return

        retrieval_start = time.time()
        chunks_data = RUNTIME.retriever.retrieve(clean_message, top_k=TOP_K)
        retrieval_time = time.time() - retrieval_start
        if not chunks_data or all(chunk["score"] < MIN_RELEVANCE_SCORE for chunk in chunks_data):
            yield (
                "No sufficiently relevant legal documents were found for this question. "
                "Try naming the act, section, or legal topic more explicitly."
            )
            return

        # Build the source footnote up front — before streaming — so we can
        # always append it even if the stream raises partway through.
        sources_line = "\n\n---\n*Sources: " + " · ".join(
            dict.fromkeys(
                chunk["metadata"].get("source_file", "?").replace(".pdf", "")
                for chunk in chunks_data
            )
        ) + "*"

        yield "⚖️ Searching legal documents..."
        messages = build_legal_messages(clean_message, chunks_data)

        current = ""
        try:
            for partial in stream_provider_reply(
                provider_name, model_name, user_api_key, messages, temperature=0.1
            ):
                current = partial
                yield current
        except Exception as stream_exc:
            # Surface the provider error but still show sources so the
            # user knows which documents were consulted.
            err_msg = safe_error_message(stream_exc)
            if current:
                yield current.strip() + "\n\n" + err_msg + sources_line
            else:
                yield err_msg + sources_line
            return

        # Happy path: append sources to the completed answer.
        yield current.strip() + sources_line
    except Exception as exc:
        yield safe_error_message(exc)


def _component_kwargs(factory, **kwargs):
    supported = inspect.signature(factory.__init__).parameters
    return {key: value for key, value in kwargs.items() if key in supported}


def _queue_kwargs(blocks, **kwargs):
    supported = inspect.signature(blocks.queue).parameters
    return {key: value for key, value in kwargs.items() if key in supported}


def build_demo() -> gr.Blocks:
    css = """
    body, .gradio-container {
        background: #f0f2f8 !important;
        font-family: 'Inter', 'Segoe UI', sans-serif;
    }

    /* ── Header ── */
    #app-header {
        background: linear-gradient(135deg, #1a237e 0%, #3949ab 100%);
        border-radius: 14px;
        padding: 24px 28px 18px;
        margin-bottom: 14px;
    }
    #app-header h1 {
        color: #fff !important;
        font-size: 1.65rem;
        font-weight: 700;
        margin: 0 0 4px;
    }
    #app-header p {
        color: #c5cae9 !important;
        font-size: 0.88rem;
        margin: 0;
    }

    /* ── Settings accordion ── */
    #settings-acc > .label-wrap {
        background: #fff !important;
        border: 1px solid #dde2f0 !important;
        border-radius: 10px !important;
        padding: 10px 16px !important;
        font-weight: 600;
        font-size: 0.92rem;
        color: #1a237e;
        margin-bottom: 10px;
        box-shadow: 0 1px 3px rgba(0,0,0,0.04);
    }
    #settings-acc > .label-wrap:hover { background: #f5f6ff !important; }
    #settings-acc .inner {
        background: #fff;
        border: 1px solid #dde2f0;
        border-top: none;
        border-radius: 0 0 10px 10px;
        padding: 14px 16px 16px;
        margin-bottom: 10px;
    }

    /* ── Test button ── */
    #test-btn {
        background: #3949ab !important;
        color: #fff !important;
        border: none !important;
        border-radius: 8px !important;
        font-weight: 600 !important;
        margin-top: 4px;
    }
    #test-btn:hover { background: #283593 !important; }

    /* ── Key result ── */
    #key-result {
        background: #f8f9ff;
        border: 1px solid #dde2f0;
        border-radius: 8px;
        padding: 8px 12px;
        font-size: 0.84rem;
        margin-top: 4px;
    }

    /* ── ChatInterface outer shell ── */
    #chat-ui {
        background: #fff;
        border: 1px solid #dde2f0;
        border-radius: 14px;
        overflow: hidden;
        box-shadow: 0 1px 6px rgba(0,0,0,0.06);
        margin-top: 4px;
    }

    /* ── Send button — target every variant Gradio uses ── */
    #chat-ui button.submit,
    #chat-ui button[aria-label="Submit"],
    #chat-ui footer button:first-child,
    #chat-ui .submit-btn {
        background: #3949ab !important;
        color: #fff !important;
        border: none !important;
        border-radius: 8px !important;
        font-weight: 600 !important;
    }
    #chat-ui button.submit:hover,
    #chat-ui button[aria-label="Submit"]:hover {
        background: #283593 !important;
    }

    /* ── Input textbox ── */
    #chat-ui textarea {
        border-radius: 8px !important;
        border: 1px solid #dde2f0 !important;
        font-size: 0.95rem !important;
    }
    #chat-ui textarea:focus {
        border-color: #3949ab !important;
        box-shadow: 0 0 0 2px rgba(57,73,171,0.15) !important;
    }

    /* ── Examples ── */
    .examples-holder { margin-top: 8px; }
    """

    PROVIDER_HELP = {
        "openai":  "e.g. gpt-4o · gpt-4o-mini · gpt-3.5-turbo",
        "groq":    "e.g. llama3-70b-8192 · mixtral-8x7b-32768 · gemma2-9b-it",
        "gemini":  "e.g. gemini-1.5-flash · gemini-1.5-pro",
    }

    with gr.Blocks(
        title="⚖️ Indian Legal AI Assistant",
        css=css,
        theme=gr.themes.Soft(primary_hue="indigo"),
    ) as demo:

        # ── Header ────────────────────────────────────────────────────────────
        gr.HTML("""
        <div id="app-header">
          <h1>⚖️ Indian Legal AI Assistant</h1>
          <p>RAG-powered Q&amp;A over Indian legal documents — BNS · BNSS · POCSO · NDPS · IT Act and more</p>
        </div>
        """)

        # ── Settings accordion (collapsed) ────────────────────────────────────
        with gr.Accordion("⚙️  Model & API Settings — click to expand", open=False, elem_id="settings-acc"):
            provider = gr.Dropdown(
                choices=list(ALLOWED_PROVIDERS),
                value=DEFAULT_PROVIDER,
                label="Provider",
            )
            model_hint = gr.Markdown(
                f"<small style='color:#666'>{PROVIDER_HELP[DEFAULT_PROVIDER]}</small>"
            )
            model_name = gr.Textbox(
                value=DEFAULT_MODEL_NAME,
                label="Model name",
                placeholder="Type the exact model string for your provider",
            )
            user_api_key = gr.Textbox(
                value="",
                type="password",
                label="API Key",
                placeholder="Paste your key here — never stored or logged",
            )
            test_key_button = gr.Button("🔍 Test Connection", elem_id="test-btn")
            key_test_status = gr.Markdown(
                "<small>Enter key + model name, then click Test Connection.</small>",
                elem_id="key-result",
            )

        # ── Chat UI ───────────────────────────────────────────────────────────
        # IMPORTANT: Do NOT pass chatbot= or textbox= as custom kwargs here.
        # When pre-built components are passed inside gr.Blocks, Gradio renders
        # them in the layout AND creates internal copies — the send button ends
        # up attached to the hidden internal copy, not the visible one.
        # Letting ChatInterface own everything fixes the missing send button.
        with gr.Group(elem_id="chat-ui"):
            ci = gr.ChatInterface(
                fn=chat,
                type="messages",
                additional_inputs=[provider, model_name, user_api_key],
                # Hide the built-in additional_inputs drawer — we have our own accordion above
                additional_inputs_accordion=gr.Accordion(visible=False, open=False),
                title=None,
                description=None,
                submit_btn="Send ➤",
                stop_btn="⏹ Stop",
                show_progress="minimal",
                save_history=False,
                flagging_mode="never",
                api_name=False,
            )

        # ── Example questions ─────────────────────────────────────────────────
        gr.Examples(
            examples=[
                ["What does POCSO stand for and what does it cover?"],
                ["Explain Section 27 of the NDPS Act"],
                ["What are the cybercrime provisions under the IT Act?"],
                ["What is the punishment for robbery under BNS?"],
                ["What are bail conditions under BNSS?"],
            ],
            inputs=ci.textbox,
            label="💡 Example questions — click any to fill the input",
            elem_id="examples-holder",
        )

        # ── Event wiring ──────────────────────────────────────────────────────
        def _update_hint(prov: str) -> str:
            return f"<small style='color:#666'>{PROVIDER_HELP.get(prov, '')}</small>"

        provider.change(fn=_update_hint, inputs=provider, outputs=model_hint)

        test_key_button.click(
            fn=test_api_connection,
            inputs=[provider, model_name, user_api_key],
            outputs=key_test_status,
        )

    demo.queue(
        **_queue_kwargs(
            demo,
            api_open=False,
            max_size=QUEUE_MAX_SIZE,
            default_concurrency_limit=DEFAULT_CONCURRENCY_LIMIT,
        )
    )
    return demo
demo = build_demo()


# %% [markdown]
# Cell 8: Launch
#
# FIX 5: `argparse.parse_args()` reads `sys.argv`, which in a Jupyter / Kaggle
#         kernel contains the kernel launch arguments (e.g. `-f kernel.json`).
#         This made the notebook crash with "unrecognised arguments" every time.
#         `parse_known_args()` silently ignores unknown arguments so the notebook
#         runs cleanly whether invoked as a script or as a kernel.

# %%
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Run the Kaggle all-in-one legal assistant.")
    parser.add_argument("--no-share", action="store_true", help="Disable the public Gradio share link.")
    parser.add_argument("--port", type=int, default=7860)
    # FIX 5: use parse_known_args so Jupyter kernel flags do not crash the notebook.
    args, _ = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_args()
    demo.launch(
        share=not args.no_share and LAUNCH_SHARE_LINK,
        inline=True,
        debug=True,
        show_error=True,
        quiet=False,
        server_name="0.0.0.0",
        server_port=args.port,
    )


if __name__ == "__main__":
    main()

Environment detected: colab
GPU available: True
All packages already installed.
Embedding model device: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_794/1230426549.py:1470: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_794/1230426549.py:1470: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://71b070971261686051.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://71b070971261686051.gradio.live
